In [1]:
# ============================================
# DEEPFAKE DETECTION - T=32 TRAINING PIPELINE
# ============================================
# 4 ablation experiments: full, spatial, temporal, sequential
# Key difference from T=16: creates NEW split (T=32 dataset has fewer videos)
#
# Input dataset: dfdc02-preprocessed-t32
# GPU: P100 recommended (16GB)
# Time: ~8-10 hours

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
    print('Project cloned.')
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'],
                   check=True)
    print('Project updated.')

# Step 3: Write training script
train_script = r'''
import os, sys, time, json, gc
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import numpy as np
print(f'numpy version: {np.__version__}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU!"}')
assert torch.cuda.is_available(), 'No GPU detected!'
print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# =============================================
# Find T=32 dataset
# =============================================
print('\n=== Searching for T=32 dataset ===')
data_path = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'real' in dirs and 'fake' in dirs:
        real_p = os.path.join(root, 'real')
        fake_p = os.path.join(root, 'fake')
        rc = len([d for d in os.listdir(real_p) if os.path.isdir(os.path.join(real_p, d))])
        fc = len([d for d in os.listdir(fake_p) if os.path.isdir(os.path.join(fake_p, d))])
        if rc > 0 and fc > 0:
            # Verify this is T=32 data by checking frame count
            sample_dir = None
            for label in ['real', 'fake']:
                lp = os.path.join(root, label)
                for vd in os.listdir(lp):
                    vdp = os.path.join(lp, vd)
                    if os.path.isdir(vdp):
                        sample_dir = vdp
                        break
                if sample_dir:
                    break
            if sample_dir:
                frame_count = len([f for f in os.listdir(sample_dir) if f.endswith('.jpg') or f.endswith('.png')])
                print(f'Found: {root} (real={rc}, fake={fc})')
                print(f'Dataset: {root}')
                print(f'Real: {rc}, Fake: {fc}, Total: {rc + fc}')
                print(f'Sample video frames: {frame_count} (expected ~32)')
                data_path = root
                break

if data_path is None:
    print('ERROR: Dataset not found! Listing /kaggle/input/:')
    for root, dirs, files in os.walk('/kaggle/input'):
        level = root.replace('/kaggle/input', '').count(os.sep)
        if level < 4:
            indent = '  ' * level
            print(f'{indent}{os.path.basename(root)}/ ({len(dirs)} dirs, {len(files)} files)')
    sys.exit(1)

# =============================================
# Run 4 experiments
# =============================================
from config import Config
from train import train
from utils import save_metrics

EXPERIMENTS = [
    {'name': 'A1_full_model',    'model_type': 'full',       'fusion_type': 'adaptive'},
    {'name': 'A2_spatial_only',  'model_type': 'spatial',    'fusion_type': 'adaptive'},
    {'name': 'A3_temporal_only', 'model_type': 'temporal',   'fusion_type': 'adaptive'},
    {'name': 'A4_sequential',    'model_type': 'sequential', 'fusion_type': 'adaptive'},
]

SPLIT_FILENAME = 'split_seed42_t32.json'

all_results = []
for i, exp in enumerate(EXPERIMENTS, 1):
    print(f'\n{"="*70}')
    print(f'[{i}/4] {exp["name"]} (model_type={exp["model_type"]}, T=32)')
    print(f'{"="*70}\n')

    cfg = Config()
    cfg.dataset_root = data_path
    cfg.dataset_name = 'dfdc02'
    cfg.model_type = exp['model_type']
    cfg.fusion_type = exp['fusion_type']
    cfg.num_frames = 32
    cfg.min_frames_per_video = 24
    cfg.max_epochs = 30
    cfg.batch_size = 8
    cfg.patience = 7
    cfg.device = 'auto'
    cfg.use_amp = True
    cfg.num_workers = 2
    cfg.pin_memory = True
    cfg.splits_dir = './splits'
    cfg.output_dir = './experiments'
    cfg.seed = 42
    cfg.unfreeze_last_n_blocks = 2
    cfg.split_filename = SPLIT_FILENAME

    # CRITICAL: First experiment creates NEW split for T=32 data,
    # subsequent experiments reuse it
    if i == 1:
        cfg.split_mode = 'random'   # Create new split
        cfg.save_split = True
    else:
        cfg.split_mode = 'fixed'    # Reuse split from experiment 1
        cfg.save_split = False

    t0 = time.time()
    try:
        metrics = train(cfg)
        metrics['experiment_name'] = exp['name']
        metrics['status'] = 'success'
        metrics['duration_min'] = round((time.time() - t0) / 60, 1)
        all_results.append(metrics)
        test = metrics.get('test', {})
        print(f'\n[OK] {exp["name"]} done in {metrics["duration_min"]} min')
        print(f'     AUC={test.get("auc",0):.4f}  Acc={test.get("accuracy",0):.4f}  F1={test.get("f1",0):.4f}')
    except Exception as e:
        elapsed = round((time.time() - t0) / 60, 1)
        print(f'\n[FAIL] {exp["name"]} after {elapsed} min: {e}')
        import traceback; traceback.print_exc()
        all_results.append({
            'experiment_name': exp['name'],
            'status': 'failed',
            'error': str(e),
            'duration_min': elapsed,
        })

    # Cleanup between experiments
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print(f'GPU Memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, '
          f'{torch.cuda.memory_reserved()/1e9:.2f} GB reserved')

    save_metrics(all_results, './experiments/all_results_t32.json')

# =============================================
# Summary
# =============================================
print(f'\n{"="*70}')
print('RESULTS SUMMARY (T=32)')
print(f'{"="*70}')
print(f'{"Experiment":<25} {"AUC":>8} {"Acc":>8} {"F1":>8} {"EER":>8} {"Epoch":>6} {"Time":>8}')
print('-' * 75)
for r in all_results:
    if r.get('status') == 'success':
        t = r.get('test', {})
        be = r.get('best_epoch', '?')
        print(f'{r["experiment_name"]:<25} {t.get("auc",0):>8.4f} {t.get("accuracy",0):>8.4f} '
              f'{t.get("f1",0):>8.4f} {t.get("eer",0):>8.4f} {be:>6} {r.get("duration_min",0):>7.1f}m')
    else:
        print(f'{r["experiment_name"]:<25} {"FAILED":>8} {r.get("error","")[:40]}')

# Package results for download
os.makedirs('/kaggle/working/experiments', exist_ok=True)
os.system('cp -r experiments/ /kaggle/working/experiments/ 2>/dev/null')
os.system('tar czf /kaggle/working/results_t32.tar.gz -C /kaggle/working/project experiments/ splits/ 2>/dev/null')
print(f'\nresults_t32.tar.gz ready for download')
'''

with open('/kaggle/working/run_training_t32.py', 'w') as f:
    f.write(train_script)

# Step 4: Run training in FRESH subprocess
print('\n=== Starting T=32 training in subprocess ===')
result = subprocess.run(
    [sys.executable, '/kaggle/working/run_training_t32.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
)
print(f'\nTraining subprocess exited with code: {result.returncode}')

# Copy results
if os.path.exists('/kaggle/working/project/experiments'):
    subprocess.run(['cp', '-r', '/kaggle/working/project/experiments/', '/kaggle/working/experiments/'])
    print('Results copied to /kaggle/working/experiments/')
if os.path.exists('/kaggle/working/results_t32.tar.gz'):
    print('results_t32.tar.gz is ready for download')

=== Installing dependencies ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 75.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 755.5/755.5 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

Dependencies installed.


Cloning into '/kaggle/working/project'...


Project cloned.

=== Starting T=32 training in subprocess ===


...


[OK] A2_spatial_only done in 140.6 min
     AUC=0.9890  Acc=0.9508  F1=0.9481
GPU Memory after cleanup: 0.02 GB allocated, 0.10 GB reserved

[3/4] A3_temporal_only (model_type=temporal, T=32)

[ДАННЫЕ] Всего видео: 3240 (real: 1699, fake: 1541)
[SPLIT] loaded fixed split: train=2267, val=485, test=488


2026-03-21 17:42:11 | INFO |   [batch 28/283] avg_loss=0.7581
2026-03-21 17:42:32 | INFO |   [batch 56/283] avg_loss=0.7454
2026-03-21 17:42:52 | INFO |   [batch 84/283] avg_loss=0.7379
2026-03-21 17:43:13 | INFO |   [batch 112/283] avg_loss=0.7463
2026-03-21 17:43:35 | INFO |   [batch 140/283] avg_loss=0.7435
2026-03-21 17:43:55 | INFO |   [batch 168/283] avg_loss=0.7446
2026-03-21 17:44:17 | INFO |   [batch 196/283] avg_loss=0.7428
2026-03-21 17:44:39 | INFO |   [batch 224/283] avg_loss=0.7448
2026-03-21 17:45:01 | INFO |   [batch 252/283] avg_loss=0.7481
2026-03-21 17:45:23 | INFO |   [batch 280/283] avg_loss=0.7477
2026-03-21 17:46:00 | INFO | Epoch 01/30 | Train Loss: 0.7470 | Train AUC: 0.4733 | Val Loss: 0.7085 | Val AUC: 0.5265 | Val Acc: 0.4969 | Val F1: 0.6269 | LR backbone: 0.000020 | LR head: 0.000060 | Time: 251.1s
2026-03-21 17:46:00 | INFO | Лучшая модель сохранена: auc = 0.5265
2026-03-21 17:46:21 | INFO |   [batch 28/283] avg_loss=0.7108
2026-03-21 17:46:42 | INFO |   


[OK] A3_temporal_only done in 125.0 min
     AUC=0.9761  Acc=0.9283  F1=0.9227
GPU Memory after cleanup: 0.02 GB allocated, 0.10 GB reserved

[4/4] A4_sequential (model_type=sequential, T=32)

[ДАННЫЕ] Всего видео: 3240 (real: 1699, fake: 1541)
[SPLIT] loaded fixed split: train=2267, val=485, test=488


2026-03-21 19:47:04 | INFO |   [batch 28/283] avg_loss=0.7326
2026-03-21 19:47:27 | INFO |   [batch 56/283] avg_loss=0.7282
2026-03-21 19:47:49 | INFO |   [batch 84/283] avg_loss=0.7356
2026-03-21 19:48:10 | INFO |   [batch 112/283] avg_loss=0.7355
2026-03-21 19:48:31 | INFO |   [batch 140/283] avg_loss=0.7336
2026-03-21 19:48:52 | INFO |   [batch 168/283] avg_loss=0.7327
2026-03-21 19:49:13 | INFO |   [batch 196/283] avg_loss=0.7290
2026-03-21 19:49:34 | INFO |   [batch 224/283] avg_loss=0.7271
2026-03-21 19:49:55 | INFO |   [batch 252/283] avg_loss=0.7265
2026-03-21 19:50:17 | INFO |   [batch 280/283] avg_loss=0.7251
2026-03-21 19:50:57 | INFO | Epoch 01/30 | Train Loss: 0.7255 | Train AUC: 0.4987 | Val Loss: 0.7047 | Val AUC: 0.5438 | Val Acc: 0.4990 | Val F1: 0.6074 | LR backbone: 0.000020 | LR head: 0.000060 | Time: 256.3s
2026-03-21 19:50:57 | INFO | Лучшая модель сохранена: auc = 0.5438
2026-03-21 19:51:20 | INFO |   [batch 28/283] avg_loss=0.7012
2026-03-21 19:51:40 | INFO |   


[OK] A4_sequential done in 125.5 min
     AUC=0.9750  Acc=0.9344  F1=0.9301
GPU Memory after cleanup: 0.02 GB allocated, 0.10 GB reserved

RESULTS SUMMARY (T=32)
Experiment                     AUC      Acc       F1      EER  Epoch     Time
---------------------------------------------------------------------------
A1_full_model               0.9869   0.9570   0.9552   0.0450     26   175.3m
A2_spatial_only             0.9890   0.9508   0.9481   0.0532     24   140.6m
A3_temporal_only            0.9761   0.9283   0.9227   0.0677     30   125.0m
A4_sequential               0.9750   0.9344   0.9301   0.0659     20   125.5m

results_t32.tar.gz ready for download

Training subprocess exited with code: 0
Results copied to /kaggle/working/experiments/
results_t32.tar.gz is ready for download
